In [1]:
import json
import pandas as pd
from tqdm import tqdm
import pickle
import random
import bm25s
import os
random.seed(42)

/projects/iris/rferreir/.envs/spe/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_json(path):
    with open(path, 'r') as f:
        data = json.load(f)
    return data

In [3]:
cities = load_json('/projects/iris/rferreir/datasets/TourismQA/original_data/final_cities_lat_long.json')
knowlegde = load_json('/projects/iris/rferreir/datasets/TourismQA/original_data/TourQue_Knowledge_SelSum.json')
with open("/projects/iris/rferreir/datasets/TourismQA/original_data/city_pois.pkl", 'rb') as f:
    city_pois = pickle.load(f)

for c in city_pois:
    tmp = {}
    tmp['R'] = set()
    tmp['H'] = set()
    tmp['A'] = set()
    for idx in city_pois[c]:
        for letter in ['R', 'H', 'A']:
            if letter in idx:
                tmp[letter].add(idx)
                break
    city_pois[c] = tmp
entities_ids = set(knowlegde.keys())

In [4]:
map_city = {}
for city in cities:
    map_city[city['city_id']] = {"coord":(city['location']['lat'], city['location']['lng']), "name":city['city_name']}

In [5]:
print(knowlegde['0_R_7327'])

{'review': ['Verdict: A great place to grab a beer and a warm pretzel at the same time.\n', 'Pros: A great bar to sit down and have a beer in peace and quiet .\n', 'Cons: The bar may be too large for some people to comfortably sit at the end of the bar, but this is a testament to the quality of the staff and the fact that it is a good bar to get a drink at .\n'], 'lat_long': [40.7201861, -73.9846227], 'name': 'Donnybrook, 35 Clinton St, New York City, NY 10002-2426', 'cluster_map': {}}


In [6]:
def sample_negatives_with_distance(q_data, idx, city_name):
    poi_type = ""
    for letter in ['R', 'H', 'A']:
        if letter in idx:
            poi_type = letter
    positives = set(q_data['all_answer_entities'])
    negatives = entities_ids.difference(positives).difference(city_pois[city_name][poi_type])
    
    # Get 2 negatives by sampling 
    easy_neg = random.sample(list(negatives), k=2)

    #Get hard negative : poi located in the same city
    hard_neg = random.sample(list(city_pois[city_name][poi_type].difference(positives)), k=1)

    return easy_neg, hard_neg

def sample_negatives_with_bm25(q_data, idx, city_name):
    poi_type = ""
    for letter in ['R', 'H', 'A']:
        if letter in idx:
            poi_type = letter
    positives = set(q_data['all_answer_entities'])
    negatives = entities_ids.difference(positives).intersection(city_pois[city_name][poi_type])

    query = q_data['question']

    query_tokens = bm25s.tokenize(query)

    retriever = retrievers[city_name][poi_type]

    results, _ = retriever.retrieve(query_tokens, k=1)

    #Get hard negative : poi located in the same city
    hard_neg = [results[0,0]['text'].split(";;")[0]]
    
    # Get 2 negatives by sampling in the same city
    easy_neg = random.sample(list(negatives.difference(set(hard_neg))), k=2)

    return easy_neg, hard_neg

In [7]:
def build_choices(choices):
    pos = choices[0]
    random.shuffle(choices)
    pos_idx = choices.index(pos)
    choices_dict = {}

    letters = ['A', 'B', 'C', 'D']

    for i, l in enumerate(letters):
        d = {}
        k = knowlegde[choices[i]]
        split = k['name'].replace("[", "").replace("]", "").split(',')
        d['name'] = split[0]
        d['adress'] = " ".join(split[1:])
        d['review'] = ' '.join(k['review'])
        if 'lat_long' in k:
            d['lat_long'] = k['lat_long']
            
        choices_dict[l] = d

    return choices_dict, letters[pos_idx]

In [8]:
def complete_data(data, sample_neg):
    new_data = []
    i = 0
    step = 100
    while i < len(data):
        q = data[i]
        city = map_city[int(q['city'])]
        for idx in q['all_answer_entities']:
            if idx in knowlegde:
                q1 = {}
                q1["question"] = q['question']
                q1['city'] = city
                #easy_neg, hard_neg = sample_negatives_with_distance(q, idx, city['name'])
                easy_neg, hard_neg = sample_neg(q, idx, city['name'])
                choices = [idx] + easy_neg + hard_neg
                choices_dict, gold = build_choices(choices)
                q1["answer"] = gold
                q1 = {**q1, **choices_dict}
                q1['tagged_locations'] = q['tagged_locations']
                q1['tagged_locations_lat_long'] = q['tagged_locations_lat_long']
                new_data.append(q1)
        i += len(q['all_answer_entities'])
        if i > step:
            print(f"{i}/{len(data)}", end="  ")
            step += 500
    return new_data

## Sample with distance

In [9]:
test_data = load_json("/projects/iris/rferreir/datasets/TourismQA/original_data/test.json")
new_data = complete_data(test_data, sample_negatives_with_distance)
print(new_data[0])

102/4342  206/4342  306/4342  405/4342  502/4342  604/4342  703/4342  810/4342  902/4342  1001/4342  1103/4342  1202/4342  1301/4342  1401/4342  1501/4342  1601/4342  1701/4342  1801/4342  1901/4342  2001/4342  2102/4342  2204/4342  2301/4342  2401/4342  2501/4342  2603/4342  2701/4342  2801/4342  2904/4342  3001/4342  3103/4342  3203/4342  3301/4342  3403/4342  3504/4342  3603/4342  3706/4342  3803/4342  3903/4342  4006/4342  4101/4342  4201/4342  4301/4342  {'question': "Hi, everyone! We (I and my wife) will be seeing Les Misérables at Queens Theatre in London. As the play ends around at 22:00 (it's so late!), we are planning to have dinner before the show. We have to arrive at the theatre by 19:00. What restaurants do you recommend? We would like to have dinner near Queens Theatre. The budget is about ￡20-30 per person. We're looking forward to your kind and useful suggestions. P. S. Is it dangerous to walk around Queen's theatre after the show? In fact, we are living in Japan, whic

In [11]:
print(len(new_data))

4298


In [12]:
with open("/projects/iris/rferreir/datasets/TourismQA/test.json", 'w') as f:
    json.dump(new_data, f)

In [14]:
train_data = load_json("/projects/iris/rferreir/datasets/TourismQA/original_data/train.json")
new_data = complete_data(train_data, sample_negatives_with_distance)
print(len(new_data))

101/43546  601/43546  1101/43546  1601/43546  2101/43546  2603/43546  3101/43546  3601/43546  4101/43546  4601/43546  5102/43546  5605/43546  6101/43546  6601/43546  7102/43546  7602/43546  8102/43546  8601/43546  9104/43546  9614/43546  10101/43546  10604/43546  11103/43546  11602/43546  12102/43546  12603/43546  13105/43546  13601/43546  14103/43546  14601/43546  15103/43546  15602/43546  16101/43546  16603/43546  17103/43546  17607/43546  18101/43546  18607/43546  19101/43546  19607/43546  20101/43546  20601/43546  21102/43546  21602/43546  22101/43546  22602/43546  23103/43546  23602/43546  24101/43546  24603/43546  25102/43546  25605/43546  26103/43546  26605/43546  27102/43546  27601/43546  28101/43546  28602/43546  29101/43546  29604/43546  30104/43546  30604/43546  31101/43546  31601/43546  32101/43546  32603/43546  33102/43546  33601/43546  34101/43546  34601/43546  35101/43546  35605/43546  36105/43546  36604/43546  37101/43546  37603/43546  38102/43546  38603/43546  39101/43

In [15]:
with open("/projects/iris/rferreir/datasets/TourismQA/train.json", 'w') as f:
    json.dump(new_data, f)

In [16]:
dev_data = load_json("/projects/iris/rferreir/datasets/TourismQA/original_data/valid.json")
new_data = complete_data(dev_data, sample_negatives_with_distance)
print(len(new_data))

107/4196  601/4196  1102/4196  1601/4196  2103/4196  2601/4196  3102/4196  3601/4196  4102/4196  4159


In [17]:
with open("/projects/iris/rferreir/datasets/TourismQA/valid.json", 'w') as f:
    json.dump(new_data, f)

## Sample with BM25

In [9]:
retrievers = {}
for c in city_pois:
    retrievers[c] = {}
    for letter in ['R', 'A', 'H']:
        path = f"/projects/iris/rferreir/datasets/TourismQA/original_data/indexes/{c}_{letter}_index_bm25"
        if os.path.exists(path):
            retrievers[c][letter] = bm25s.BM25.load(path, load_corpus=True)

In [10]:
test_data = load_json("/projects/iris/rferreir/datasets/TourismQA/original_data/test.json")
new_data = complete_data(test_data, sample_negatives_with_bm25)
print(new_data[0])
print(len(new_data))

102/4342  

604/4342  

1103/4342  

1601/4342  

2102/4342  

2603/4342  

3103/4342  

3603/4342  

4101/4342  

{'question': "Hi, everyone! We (I and my wife) will be seeing Les Misérables at Queens Theatre in London. As the play ends around at 22:00 (it's so late!), we are planning to have dinner before the show. We have to arrive at the theatre by 19:00. What restaurants do you recommend? We would like to have dinner near Queens Theatre. The budget is about ￡20-30 per person. We're looking forward to your kind and useful suggestions. P. S. Is it dangerous to walk around Queen's theatre after the show? In fact, we are living in Japan, which is a very secure country. We are not sure whether London is a safe city. When we travelled in New York, we were advised not to go out after midnight.", 'city': {'coord': (51.5073509, -0.1277583), 'name': 'London'}, 'answer': 'D', 'A': {'name': 'The Georgian', 'adress': ' 27 Balham Hill  Clapham South  London SW12 9DX', 'review': "Verdict: If you're looking for a traditional Georgian breakfast, this is the one to get if you want a taste of Georgian cuisine.\n

In [11]:
with open("/projects/iris/rferreir/datasets/TourismQA/test_bm25.json", 'w') as f:
    json.dump(new_data, f)

In [ ]:
train_data = load_json("/projects/iris/rferreir/datasets/TourismQA/original_data/train.json")
new_data = complete_data(train_data, sample_negatives_with_bm25)
print(len(new_data))

101/43546  

601/43546  

1101/43546  

1601/43546  

2101/43546  

...

10101/43546  

10604/43546  

11103/43546  

11602/43546  

12102/43546  

In [ ]:
with open("/projects/iris/rferreir/datasets/TourismQA/train_bm25.json", 'w') as f:
    json.dump(new_data, f)

In [ ]:
dev_data = load_json("/projects/iris/rferreir/datasets/TourismQA/original_data/valid.json")
new_data = complete_data(dev_data, sample_negatives_with_bm25)
print(len(new_data))

In [ ]:
with open("/projects/iris/rferreir/datasets/TourismQA/valid_bm25.json", 'w') as f:
    json.dump(new_data, f)